In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

c:\Users\UCX37\AppData\Local\miniconda3\envs\intent-bert\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
chunks_df = pd.read_parquet(
    "../processed/chunks.parquet"
)

print(chunks_df.shape)

chunks_df.head()

(48, 9)


,chunk_id,document_id,filename,file_type,tenant,category,subcategory,chunk_index,chunk_text
0,b66130c0-3601-446c-abb6-71bae4a21dd0,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,0,Call Centre Standard Operating Procedures - 20...
1,ff395496-839f-449d-be58-ee05f149a7dc,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,1,"peak periods, a staff compliment of at least 1..."
2,5ad64fe9-a281-4222-805a-645b602bdf55,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,2,and abandoned The fortnightly summary historic...
3,041d7c16-2705-4777-b176-5d10ecdb080d,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,3,number before transferring. 9. The Agent will ...
4,e4935a1e-5195-42b5-9465-8c3e3aba8a1a,734bea56-0491-428b-8713-b1be028e6a3d,CIB_CW_09_01_19.pdf,.pdf,None,None,None,0,CIB/CW/09/01/19 Claims workflow and documentat...


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-mpnet-base-v2",
    local_files_only=True
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2198.09it/s]


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    local_files_only=True
)

print("Model loaded successfully!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2862.04it/s]

Model loaded successfully!


In [5]:
texts = chunks_df["chunk_text"].tolist()

embeddings = model.encode(

    texts,

    batch_size=16,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True

)

Batches: 100%|██████████| 3/3 [00:52<00:00, 17.59s/it]


In [6]:
print(embeddings.shape)

print(embeddings[0][:10])

(48, 768)
[ 0.04981226 -0.08564965 -0.00730773 -0.01846929 -0.03831366  0.02555521
  0.0287549  -0.04154808  0.00036322  0.00504805]


In [7]:
chunks_df["embedding_index"] = range(len(chunks_df))

chunks_df.head()

,chunk_id,document_id,filename,file_type,tenant,category,subcategory,chunk_index,chunk_text,embedding_index
0,b66130c0-3601-446c-abb6-71bae4a21dd0,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,0,Call Centre Standard Operating Procedures - 20...,0
1,ff395496-839f-449d-be58-ee05f149a7dc,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,1,"peak periods, a staff compliment of at least 1...",1
2,5ad64fe9-a281-4222-805a-645b602bdf55,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,2,and abandoned The fortnightly summary historic...,2
3,041d7c16-2705-4777-b176-5d10ecdb080d,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,3,number before transferring. 9. The Agent will ...,3
4,e4935a1e-5195-42b5-9465-8c3e3aba8a1a,734bea56-0491-428b-8713-b1be028e6a3d,CIB_CW_09_01_19.pdf,.pdf,None,None,None,0,CIB/CW/09/01/19 Claims workflow and documentat...,4


In [8]:
np.save(

    "../processed/embeddings.npy",

    embeddings

)

chunks_df.to_parquet(

    "../processed/chunks_with_embeddings.parquet",

    index=False

)